# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# constants
MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'

In [6]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OpenAI API Key was correctly setup.")
else:
    print("OpenAPI API Key is missing.")

openai = OpenAI()

technical_explainer_system_prompt = """
You will be asked code technical questions.
You will be provided with snippets of code that can be
the body of a function or the body of class,
explain every line and what it does,
remembering the context from the previous lines.
Respond in markdown without html tags.
"""

def messages_for(system_prompt, user_question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]


OpenAI API Key was correctly setup.


In [4]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = soup.find_all("a")
    links = [link.get("href") for link in links if link.get("href")]
    return links
"""

In [7]:
# Get gpt-4o-mini to answer, with streaming
def stream_technical_answer(user_question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages_for(technical_explainer_system_prompt, user_question),
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_technical_answer(question)

Here’s what each line does and why it’s there, plus some notes on behavior and potential improvements.

- Line: `response = requests.get(url, headers=headers)`
  - What it does: Sends an HTTP GET request to the specified `url`, including any provided HTTP headers in `headers`. It returns a Response object.
  - Why: To fetch the HTML (or other content) of the target page so it can be parsed.
  - Notes: Requires that `requests` is imported. If the request fails due to network issues, timeouts, or server errors, this line may raise exceptions or return a response with a non-200 status code. No error handling is shown here.

- Line: `soup = BeautifulSoup(response.content, "html.parser")`
  - What it does: Parses the raw content of the response (bytes) into a BeautifulSoup parse tree using Python’s built-in HTML parser.
  - Why: To enable easy navigation and extraction of elements from the HTML document.
  - Notes: `response.content` provides the raw bytes; `BeautifulSoup` will decode as needed. Requires that `BeautifulSoup` (bs4) is imported.

- Line: `links = soup.find_all("a")`
  - What it does: Finds all `<a>` (anchor) elements in the parsed HTML.
  - Why: Anchor tags typically represent hyperlinks on a page, so this collects all potential links.

- Line: `links = [link.get("href") for link in links if link.get("href")]`
  - What it does: Creates a new list of the `href` attribute values for each anchor tag, but only for anchors that actually have an `href` attribute.
  - Why: To extract the actual URLs (or URL fragments) linked by each anchor, while ignoring anchors without links.
  - Notes: The resulting list contains strings like absolute URLs (e.g., "https://example.com/page") or relative URLs (e.g., "/page"). Duplicates may occur if the same link appears multiple times.

- Line: `return links`
  - What it does: Returns the list of extracted `href` values to the caller.
  - Why: To provide the caller with all discovered link targets from the page.

Overall behavior:
- The function (as shown) fetches a page, parses the HTML, collects all hyperlink targets from anchor tags, and returns a list of href strings. It does not resolve relative URLs to absolute URLs, nor does it deduplicate, nor does it validate the response status.

Potential issues and improvements:
- Error handling: Add `response.raise_for_status()` or check `response.status_code` to handle HTTP errors gracefully.
- Imports: Ensure `import requests` and `from bs4 import BeautifulSoup` are present.
- Response validation: Ensure the content is HTML before parsing (check `Content-Type`).
- Relative URLs: If you need absolute URLs, resolve them with something like `urllib.parse.urljoin(url, href)`.
- Duplicates: If you want unique links, consider deduplicating (e.g., `list(dict.fromkeys(links))` or using a set).
- Performance and robustness: For very large pages, consider streaming approaches or filtering during parsing; handle potential malformed HTML gracefully.

In [8]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages_for(technical_explainer_system_prompt, question))
technical_answer = response.choices[0].message.content

display(Markdown(technical_answer))

This code is used to extract all the links from a given URL.

Here's a breakdown of what it does:

### Importing Request Library
```python
response = requests.get(url, headers=headers)
```
*   It sends a GET request to the provided URL (`url`) using the `requests` library. The `headers` parameter is passed to the function to include the HTTP headers in the request.

*   The response from the server is stored in the `response` variable.

### Parsing HTML Response
```python
soup = BeautifulSoup(response.content, "html.parser")
```
*   It uses the ` Beautiful Soup` library to parse the HTML content of the response. This library allows us to navigate and search through the HTML content like it were a tree.

*   `BeautifulSoup` takes the content of the response and creates a representation of it (a `BeautifulSoup` object) which we then call `soup`.

### Finding All Links
```python
links = soup.find_all("a")
```
*   This line finds all the elements that are tags of type 'a' in the HTML tree. Since these elements typically contain links, this line is a good starting point to extract links from a webpage.

### Filtering Out Empty Links
```python
links = [link.get("href") for link in links if link.get("href")]
```
*   The list comprehension filters out any elements from the `links` list where the attribute 'href' is not found. The attribute 'href' is usually present in tags of type 'a' and holds the actual link to the webpage.

*   So, if there are some links that do not have a 'href' attribute they are simply ignored and not added to the `links` list.

### Returning the List of Links
```python
return links
```
*   This line returns the list of links extracted from the webpage.

Overall, this code is used to extract links from a given webpage, handle empty links, and return a list of valid links.